# Design > Randomizer

<div class="alert alert-info">Randomly assign experimental units to treatment conditions</div>

The `randomizer` function performs random assignment of units (e.g., customers, stores, participants) to experimental conditions. It supports both simple complete randomization and block randomization (where balance is enforced within groups defined by a blocking variable).

In [ ]:
import polars as pl
import pyrsm as rsm

In [ ]:
## setup pyrsm for autoreload
%reload_ext autoreload
%autoreload 2
%aimport pyrsm

# Example: Simple random assignment

Suppose you have a list of 100 people and want to randomly assign them to two conditions: "test" and "control". Each person has an equal probability of being assigned to either condition.

In [ ]:
rndnames = pl.read_parquet("../data/design/rndnames.parquet")
rndnames.head()

In [ ]:
r = rsm.design.randomizer(
    {"rndnames": rndnames},
    vars=["Names"],
    conditions=["test", "control"],
    seed=1234,
)
r.summary()

With complete randomization and equal probabilities, each condition gets exactly 50 units. The `.conditions` column is added to the data showing the assignment:

In [ ]:
r.data.head(10)

# Three or more conditions

You can assign units to any number of conditions. With 100 units and 3 conditions, each condition gets approximately 33-34 units:

In [ ]:
r3 = rsm.design.randomizer(
    rndnames,
    vars=["Names"],
    conditions=["A", "B", "C"],
    seed=1234,
)
r3.summary()

# Unequal probabilities

If you want to assign more units to one condition (e.g., 70% treatment, 30% control), use the `probs` parameter:

In [ ]:
r_unequal = rsm.design.randomizer(
    rndnames,
    vars=["Names"],
    conditions=["treatment", "control"],
    probs=[0.7, 0.3],
    seed=1234,
)
r_unequal.summary()

# Block randomization

Block randomization ensures that within each level of a blocking variable, the assignment is balanced. This is useful when you want to ensure balance across important covariates (e.g., gender).

In this example, we block on `Gender` so that within both the Female and Male groups, half are assigned to "test" and half to "control":

In [ ]:
r_block = rsm.design.randomizer(
    {"rndnames": rndnames},
    vars=["Names"],
    blocks="Gender",
    conditions=["test", "control"],
    seed=1234,
)
r_block.summary()

The cross-tabulation shows that within each gender group, the test and control conditions are perfectly balanced (25 each for both Female and Male).

© Vincent Nijs (2026)